# Silver Layer Quality — Transparency View

**Project:** ENEM Opportunity & Equity Radar  
**Stage:** CRISP-DM → Silver Layer Quality Transparency

Loads Bronze row counts, Silver final row counts, and `quality_report`; computes per-year valid/removed metrics; plots bar chart and top filtering rules table. Reproducible with Spark + matplotlib.

In [ ]:
from pathlib import Path
import sys

# Project root: run from repo root (spark) or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

ANOS = [2020, 2021, 2022, 2023, 2024]
DIR_BRONZE = ROOT / "data" / "bronze"
DIR_SILVER = ROOT / "data" / "silver"

In [ ]:
spark = (
    SparkSession.builder.appName("Silver-Quality-Viz")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

## 1. Load Bronze row counts and Silver final row counts

In [ ]:
bronze_counts = []
for ano in ANOS:
    path = DIR_BRONZE / f"enem_{ano}.parquet"
    if path.exists():
        n = spark.read.parquet(str(path)).count()
        bronze_counts.append((ano, n))
    else:
        bronze_counts.append((ano, 0))

bronze_df = spark.createDataFrame(bronze_counts, ["ano", "total_raw"])
bronze_df.show()

In [ ]:
participante_path = DIR_SILVER / "enem_participante"
if participante_path.exists():
    silver_counts = (
        spark.read.parquet(str(participante_path))
        .groupBy("ano")
        .count()
        .withColumnRenamed("count", "total_valid")
    )
    silver_counts.show()
else:
    silver_counts = spark.createDataFrame(
        [(a, 0) for a in ANOS],
        ["ano", "total_valid"]
    )

## 2. Compute per-year: total_raw, total_valid, total_removed, % removed

In [ ]:
summary = bronze_df.join(silver_counts, "ano", "left").fillna(0, subset=["total_valid"])
summary = summary.withColumn(
    "total_removed",
    F.greatest(F.col("total_raw") - F.col("total_valid"), F.lit(0))
).withColumn(
    "pct_removed",
    F.when(F.col("total_raw") > 0,
           F.round((F.col("total_removed") / F.col("total_raw")) * 100, 2)
    ).otherwise(F.lit(0.0))
)
summary.show()

## 3. Bar chart: Data Cleaning Impact — Transparency View

In [ ]:
rows = summary.orderBy("ano").collect()
years = [r["ano"] for r in rows]
valid = [r["total_valid"] for r in rows]
removed = [r["total_removed"] for r in rows]

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(years))
w = 0.35
bars1 = ax.bar([i - w/2 for i in x], valid, w, label="Valid rows", color="#2ecc71")
bars2 = ax.bar([i + w/2 for i in x], removed, w, label="Removed rows", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("Row count")
ax.set_xlabel("Year")
ax.set_title("Data Cleaning Impact — Transparency View")
ax.legend()
ax.set_ylim(0, max(valid + removed) * 1.05 if (valid + removed) else 1)
plt.tight_layout()
plt.show()

## 4. Load quality_report and aggregate by rule

In [ ]:
report_path = DIR_SILVER / "quality_report.parquet"
if report_path.exists():
    quality_df = spark.read.parquet(str(report_path))
    total_removed_overall = quality_df.agg(F.sum("rows_removed").alias("total")).collect()[0]["total"] or 0
    rule_summary = (
        quality_df
        .groupBy("rule_name")
        .agg(F.sum("rows_removed").alias("total_rows_removed"))
        .withColumn(
            "pct_of_total_removed",
            F.when(F.lit(total_removed_overall) > 0, F.round((F.col("total_rows_removed") / F.lit(total_removed_overall)) * 100, 2)).otherwise(F.lit(0.0))
        )
        .orderBy(F.desc("total_rows_removed"))
    )
    rule_summary.show(50, truncate=False)
else:
    rule_summary = None
    print("quality_report.parquet not found.")

## 5. Table: Top filtering rules by total rows removed

In [ ]:
if rule_summary is not None:
    top_rules = rule_summary.toPandas()
    top_rules.columns = ["rule_name", "total_rows_removed", "% of total_removed"]
    display(top_rules)
else:
    print("No quality report available.")

In [ ]:
spark.stop()